In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from google.colab import drive

from collections import Counter

import warnings
warnings.filterwarnings("ignore")

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DATA_PATH = "/content/drive/MyDrive/UNGA/2026_02_06_ga_voting.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 947434
Columns: 20


In [4]:
print(df.columns.tolist())

['undl_id', 'ms_code', 'ms_name', 'ms_vote', 'date', 'session', 'resolution', 'draft', 'committee_report', 'meeting', 'title', 'agenda_title', 'subjects', 'vote_note', 'total_yes', 'total_no', 'total_abstentions', 'total_non_voting', 'total_ms', 'undl_link']


In [5]:
# Convert session to numeric
df["session"] = pd.to_numeric(df["session"], errors="coerce")

# Keep only Sessions 71–79 (2016–2025)
df_2016_2025 = df[df["session"].between(71, 79)].copy()

print("Original rows:", len(df))
print("Filtered rows:", len(df_2016_2025))

print("\nSessions included:")
print(sorted(df_2016_2025["session"].unique()))

Original rows: 947434
Filtered rows: 164629

Sessions included:
[np.float64(71.0), np.float64(72.0), np.float64(73.0), np.float64(74.0), np.float64(75.0), np.float64(76.0), np.float64(77.0), np.float64(78.0), np.float64(79.0)]


In [6]:
resolutions = (
    df_2016_2025[
        ["session", "resolution", "title", "subjects"]
    ]
    .drop_duplicates()
    .sort_values(["session", "resolution"])
)

print("Total unique resolutions:", len(resolutions))

display(resolutions.head(20))

Total unique resolutions: 853


,session,resolution,title,subjects
631729,71.0,A/RES/71/102,Information from Non-Self-Governing Territorie...,NON-SELF-GOVERNING TERRITORIES--REPORTS
771171,71.0,A/RES/71/103,Economic and other activities which affect the...,NON-SELF-GOVERNING TERRITORIES--ECONOMIC INTER...
770978,71.0,A/RES/71/104,Implementation of the Declaration of the Grant...,DECOLONIZATION--UN SYSTEM
583671,71.0,A/RES/71/11,Cooperation between the United Nations and the...,LEAGUE OF ARAB STATES--UN
771750,71.0,A/RES/71/121,Dissemination of information on decolonization...,DECOLONIZATION
632115,71.0,A/RES/71/122,Implementation of the Declaration on the Grant...,DECOLONIZATION
631150,71.0,A/RES/71/130,The Situation in the Syrian Arab Republic : re...,ARMED CONFLICTS PREVENTION
770399,71.0,A/RES/71/174,Report of the Human Rights Council : resolutio...,UN. HUMAN RIGHTS COUNCIL--REPORTS
801854,71.0,A/RES/71/179,"Combating glorification of Nazism, neo-Nazism ...",RACIAL DISCRIMINATION--ELIMINATION
801661,71.0,A/RES/71/181,A global call for concrete action for the tota...,RACIAL DISCRIMINATION--PROGRAMMES OF ACTION


In [7]:
# Convert text columns to uppercase and handle missing values
resolutions["title"] = resolutions["title"].fillna("").str.upper()
resolutions["subjects"] = resolutions["subjects"].fillna("").str.upper()

# -----------------------------
# Gaza / Palestine
# -----------------------------
gaza_keywords = [
    "PALESTIN",
    "GAZA",
    "ISRAEL",
    "OCCUPIED PALESTINIAN",
    "MIDDLE EAST"
]

gaza_resolutions = resolutions[
    resolutions["title"].str.contains("|".join(gaza_keywords), regex=True)
    |
    resolutions["subjects"].str.contains("|".join(gaza_keywords), regex=True)
]

# -----------------------------
# Ukraine
# -----------------------------
ukraine_keywords = [
    "UKRAINE",
    "AGGRESSION AGAINST UKRAINE",
    "TERRITORIAL INTEGRITY OF UKRAINE"
]

ukraine_resolutions = resolutions[
    resolutions["title"].str.contains("|".join(ukraine_keywords), regex=True)
    |
    resolutions["subjects"].str.contains("|".join(ukraine_keywords), regex=True)
]

# -----------------------------
# Climate
# -----------------------------
climate_keywords = [
    "CLIMATE",
    "SUSTAINABLE DEVELOPMENT",
    "ENVIRONMENT",
    "PARIS AGREEMENT",
    "UNFCCC",
    "GLOBAL WARMING"
]

climate_resolutions = resolutions[
    resolutions["title"].str.contains("|".join(climate_keywords), regex=True)
    |
    resolutions["subjects"].str.contains("|".join(climate_keywords), regex=True)
]

print("Gaza resolutions:", len(gaza_resolutions))
print("Ukraine resolutions:", len(ukraine_resolutions))
print("Climate resolutions:", len(climate_resolutions))

Gaza resolutions: 138
Ukraine resolutions: 14
Climate resolutions: 45


In [8]:
# Aggregate
aggregate_votes = df_2016_2025.copy()

# Gaza
gaza_votes = df_2016_2025[
    df_2016_2025["resolution"].isin(gaza_resolutions["resolution"])
].copy()

# Ukraine
ukraine_votes = df_2016_2025[
    df_2016_2025["resolution"].isin(ukraine_resolutions["resolution"])
].copy()

# Climate
climate_votes = df_2016_2025[
    df_2016_2025["resolution"].isin(climate_resolutions["resolution"])
].copy()

print("Aggregate vote records:", len(aggregate_votes))
print("Gaza vote records:", len(gaza_votes))
print("Ukraine vote records:", len(ukraine_votes))
print("Climate vote records:", len(climate_votes))

Aggregate vote records: 164629
Gaza vote records: 26634
Ukraine vote records: 2702
Climate vote records: 8685


In [9]:
def create_vote_matrix(df_votes):
    matrix = (
        df_votes
        .pivot_table(
            index="ms_name",
            columns="resolution",
            values="ms_vote",
            aggfunc="first"
        )
        .sort_index()
    )
    return matrix

aggregate_matrix = create_vote_matrix(aggregate_votes)
gaza_matrix = create_vote_matrix(gaza_votes)
ukraine_matrix = create_vote_matrix(ukraine_votes)
climate_matrix = create_vote_matrix(climate_votes)

print("Aggregate:", aggregate_matrix.shape)
print("Gaza:", gaza_matrix.shape)
print("Ukraine:", ukraine_matrix.shape)
print("Climate:", climate_matrix.shape)

Aggregate: (199, 853)
Gaza: (199, 138)
Ukraine: (199, 14)
Climate: (199, 45)


In [10]:
from itertools import combinations
import pandas as pd

def calculate_agreement(vote_matrix):
    countries = vote_matrix.index.tolist()
    records = []

    for c1, c2 in combinations(countries, 2):

        v1 = vote_matrix.loc[c1]
        v2 = vote_matrix.loc[c2]

        # Remove absent votes
        valid = (
            (v1 != "X") &
            (v2 != "X") &
            (~v1.isna()) &
            (~v2.isna())
        )

        if valid.sum() == 0:
            continue

        agreement = (v1[valid] == v2[valid]).mean()

        records.append({
            "country1": c1,
            "country2": c2,
            "agreement": agreement
        })

    return pd.DataFrame(records)

In [11]:
aggregate_agreement = calculate_agreement(aggregate_matrix)
gaza_agreement = calculate_agreement(gaza_matrix)
ukraine_agreement = calculate_agreement(ukraine_matrix)
climate_agreement = calculate_agreement(climate_matrix)

print("Aggregate:", len(aggregate_agreement))
print("Gaza:", len(gaza_agreement))
print("Ukraine:", len(ukraine_agreement))
print("Climate:", len(climate_agreement))

Aggregate: 19675
Gaza: 19670
Ukraine: 18811
Climate: 19670


In [12]:
print(aggregate_matrix.isna().sum().sum())
print(gaza_matrix.isna().sum().sum())
print(ukraine_matrix.isna().sum().sum())
print(climate_matrix.isna().sum().sum())

5118
828
84
270


In [13]:
print(aggregate_votes["ms_vote"].unique())

['Y' 'X' 'A' 'N']


In [14]:
for name, matrix in [
    ("Aggregate", aggregate_matrix),
    ("Gaza", gaza_matrix),
    ("Ukraine", ukraine_matrix),
    ("Climate", climate_matrix),
]:
    print(name)
    print((matrix.notna() & (matrix != "X")).sum(axis=1).describe())
    print()

Aggregate
count    199.000000
mean     754.376884
std      150.921893
min       65.000000
25%      728.000000
50%      825.000000
75%      849.000000
max      853.000000
dtype: float64

Gaza
count    199.000000
mean     121.010050
std       29.313525
min        2.000000
25%      115.000000
50%      137.000000
75%      138.000000
max      138.000000
dtype: float64

Ukraine
count    199.000000
mean      11.572864
std        3.678146
min        0.000000
25%       10.000000
50%       14.000000
75%       14.000000
max       14.000000
dtype: float64

Climate
count    199.000000
mean      39.603015
std        8.731008
min        4.000000
25%       39.000000
50%       43.000000
75%       45.000000
max       45.000000
dtype: float64



In [15]:
import networkx as nx

THRESHOLD = 0.80

def create_graph(agreement_df, threshold=THRESHOLD):

    G = nx.Graph()

    for _, row in agreement_df.iterrows():

        if row["agreement"] >= threshold:

            G.add_edge(
                row["country1"],
                row["country2"],
                weight=row["agreement"]
            )

    return G


aggregate_G = create_graph(aggregate_agreement)
gaza_G = create_graph(gaza_agreement)
ukraine_G = create_graph(ukraine_agreement)
climate_G = create_graph(climate_agreement)

print("Aggregate:", aggregate_G.number_of_nodes(), aggregate_G.number_of_edges())
print("Gaza:", gaza_G.number_of_nodes(), gaza_G.number_of_edges())
print("Ukraine:", ukraine_G.number_of_nodes(), ukraine_G.number_of_edges())
print("Climate:", climate_G.number_of_nodes(), climate_G.number_of_edges())

Aggregate: 197 8785
Gaza: 196 9464
Ukraine: 195 4635
Climate: 198 8533


In [16]:
!pip install python-louvain

In [17]:
import community.community_louvain as community_louvain

aggregate_partition = community_louvain.best_partition(aggregate_G)
gaza_partition = community_louvain.best_partition(gaza_G)
ukraine_partition = community_louvain.best_partition(ukraine_G)
climate_partition = community_louvain.best_partition(climate_G)

print("Aggregate communities:", len(set(aggregate_partition.values())))
print("Gaza communities:", len(set(gaza_partition.values())))
print("Ukraine communities:", len(set(ukraine_partition.values())))
print("Climate communities:", len(set(climate_partition.values())))

Aggregate communities: 3
Gaza communities: 4
Ukraine communities: 4
Climate communities: 4


In [18]:
centrality = {
    "Aggregate": nx.degree_centrality(aggregate_G),
    "Gaza": nx.degree_centrality(gaza_G),
    "Ukraine": nx.degree_centrality(ukraine_G),
    "Climate": nx.degree_centrality(climate_G),
}

In [19]:
important_countries = [

"UNITED STATES",
"RUSSIAN FEDERATION",
"CHINA",

"INDIA",
"JAPAN",
"GERMANY",
"BRAZIL",

"SOUTH AFRICA",

"FRANCE",
"UNITED KINGDOM",

"CANADA",
"AUSTRALIA",

"ITALY",

"MEXICO",

"INDONESIA",

"NIGERIA",

"SAUDI ARABIA",

"TURKIYE",

"IRAN",

"ISRAEL",

"PALESTINE",

"UKRAINE",

"POLAND",

"EGYPT",

"ARGENTINA"

]

In [20]:
important_check = [
    "UNITED STATES",
    "CHINA",
    "RUSSIAN FEDERATION",
    "UNITED KINGDOM",
    "FRANCE",
    "INDIA",
    "GERMANY",
    "JAPAN",
    "BRAZIL",
    "PAKISTAN",
    "SOUTH AFRICA",
    "INDONESIA",
    "SAUDI ARABIA",
    "IRAN (ISLAMIC REPUBLIC OF)",
    "TÜRKIYE",
    "EGYPT",
    "NIGERIA",
    "MEXICO",
    "ARGENTINA",
    "CANADA",
    "AUSTRALIA",
    "ITALY",
    "REPUBLIC OF KOREA"
]

for c in important_check:
    print(c, c in aggregate_G.nodes())

UNITED STATES True
CHINA True
RUSSIAN FEDERATION False
UNITED KINGDOM True
FRANCE True
INDIA True
GERMANY True
JAPAN True
BRAZIL True
PAKISTAN True
SOUTH AFRICA True
INDONESIA True
SAUDI ARABIA True
IRAN (ISLAMIC REPUBLIC OF) True
TÜRKIYE False
EGYPT True
NIGERIA True
MEXICO True
ARGENTINA True
CANADA True
AUSTRALIA True
ITALY True
REPUBLIC OF KOREA True


In [21]:
print(len(aggregate_G.nodes()))

197


In [22]:
important_check = [
    "UNITED STATES",
    "CHINA",
    "RUSSIAN FEDERATION",
    "UNITED KINGDOM",
    "FRANCE",
    "INDIA",
    "GERMANY",
    "JAPAN",
    "BRAZIL",
    "PAKISTAN",
    "SOUTH AFRICA",
    "INDONESIA",
    "SAUDI ARABIA",
    "IRAN (ISLAMIC REPUBLIC OF)",
    "TÜRKIYE",
    "EGYPT",
    "NIGERIA",
    "MEXICO",
    "ARGENTINA",
    "CANADA",
    "AUSTRALIA",
    "ITALY",
    "REPUBLIC OF KOREA"
]

for c in important_check:
    print(c, c in aggregate_G.nodes())

UNITED STATES True
CHINA True
RUSSIAN FEDERATION False
UNITED KINGDOM True
FRANCE True
INDIA True
GERMANY True
JAPAN True
BRAZIL True
PAKISTAN True
SOUTH AFRICA True
INDONESIA True
SAUDI ARABIA True
IRAN (ISLAMIC REPUBLIC OF) True
TÜRKIYE False
EGYPT True
NIGERIA True
MEXICO True
ARGENTINA True
CANADA True
AUSTRALIA True
ITALY True
REPUBLIC OF KOREA True


In [23]:
for x in sorted(aggregate_G.nodes()):
    if "RUSS" in x.upper() or "TURK" in x.upper():
        print(x)

BRUNEI DARUSSALAM
TURKEY
TURKMENISTAN


In [24]:
for x in sorted(aggregate_G.nodes()):
    if "FED" in x.upper() or "RUS" in x.upper() or "MOS" in x.upper():
        print(x)

BELARUS
BRUNEI DARUSSALAM
CYPRUS
MICRONESIA (FEDERATED STATES OF)


In [25]:
print(df_2016_2025[df_2016_2025["ms_name"].str.contains("RUSS", case=False, na=False)]["ms_name"].unique())

['BRUNEI DARUSSALAM' 'RUSSIAN FEDERATION']


In [26]:
def create_graph(agreement_df, threshold=THRESHOLD):

    G = nx.Graph()

    for _, row in agreement_df.iterrows():

        if row["agreement"] >= threshold:

            G.add_edge(
                row["country1"],
                row["country2"],
                weight=row["agreement"]
            )

In [27]:
import networkx as nx

THRESHOLD = 0.80

def create_graph(agreement_df, all_countries, threshold=THRESHOLD):

    G = nx.Graph()

    # Add every country first
    G.add_nodes_from(all_countries)

    # Then add edges
    for _, row in agreement_df.iterrows():

        if row["agreement"] >= threshold:

            G.add_edge(
                row["country1"],
                row["country2"],
                weight=row["agreement"]
            )

    return G

In [28]:
all_countries = sorted(df_2016_2025["ms_name"].unique())

In [ ]:
aggregate_G = create_graph(aggregate_agreement, all_countries)
gaza_G = create_graph(gaza_agreement, all_countries)
ukraine_G = create_graph(ukraine_agreement, all_countries)
climate_G = create_graph(climate_agreement, all_countries)

In [ ]:
print("Aggregate:", aggregate_G.number_of_nodes())
print("Gaza:", gaza_G.number_of_nodes())
print("Ukraine:", ukraine_G.number_of_nodes())
print("Climate:", climate_G.number_of_nodes())

In [ ]:
print("RUSSIAN FEDERATION" in aggregate_G.nodes())
print("RUSSIAN FEDERATION" in gaza_G.nodes())
print("RUSSIAN FEDERATION" in ukraine_G.nodes())
print("RUSSIAN FEDERATION" in climate_G.nodes())

In [ ]:
# Layout for Aggregate Network
pos_aggregate = nx.spring_layout(
    aggregate_G,
    seed=42,
    k=0.35,
    iterations=500,
    weight="weight"
)

In [ ]:
import community as community_louvain

partition_aggregate = community_louvain.best_partition(
    aggregate_G,
    weight="weight",
    random_state=42
)

print("Communities:", len(set(partition_aggregate.values())))

In [ ]:
import community

print(community)
print(dir(community))

In [ ]:
import community as community_louvain

In [ ]:
from community import community_louvain

partition_aggregate = community_louvain.best_partition(
    aggregate_G,
    weight="weight",
    random_state=42
)

print("Communities:", len(set(partition_aggregate.values())))

In [ ]:
import community.community_louvain as community_louvain

In [ ]:
from collections import Counter

community_sizes = Counter(partition_aggregate.values())

print("Community sizes:")
for comm, size in sorted(community_sizes.items()):
    print(f"Community {comm}: {size} countries")

In [ ]:
community_members = {}

for country, comm in partition_aggregate.items():
    community_members.setdefault(comm, []).append(country)

for comm in sorted(community_members):
    print(f"\n===== Community {comm} =====")
    print(sorted(community_members[comm])[:25])   # first 25 countries

In [ ]:
from collections import Counter
import networkx as nx
from community import community_louvain

thresholds = [0.75, 0.80, 0.85]

all_countries = sorted(df_2016_2025["ms_name"].unique())

for t in thresholds:

    G = nx.Graph()
    G.add_nodes_from(all_countries)

    for _, row in aggregate_agreement.iterrows():
        if row["agreement"] >= t:
            G.add_edge(
                row["country1"],
                row["country2"],
                weight=row["agreement"]
            )

    partition = community_louvain.best_partition(
        G,
        weight="weight",
        random_state=42
    )

    modularity = community_louvain.modularity(
        partition,
        G,
        weight="weight"
    )

    sizes = Counter(partition.values())

    print("=" * 60)
    print(f"Threshold: {t}")
    print("Nodes:", G.number_of_nodes())
    print("Edges:", G.number_of_edges())
    print("Communities:", len(sizes))
    print("Modularity:", round(modularity,4))
    print("Community sizes:", sorted(sizes.values(), reverse=True))

In [ ]:
# Build aggregate graph at 0.75
aggregate_G = create_graph(aggregate_agreement, all_countries, threshold=0.75)

from community import community_louvain
partition_aggregate = community_louvain.best_partition(
    aggregate_G,
    weight="weight",
    random_state=42
)

from collections import Counter

print("Community sizes:", Counter(partition_aggregate.values()))

community_members = {}
for country, comm in partition_aggregate.items():
    community_members.setdefault(comm, []).append(country)

for comm in sorted(community_members):
    print(f"\n===== Community {comm} =====")
    print(sorted(community_members[comm])[:40])

In [ ]:
from collections import Counter
import networkx as nx
from community import community_louvain

thresholds = [0.50, 0.60, 0.70, 0.75, 0.80]

all_countries = sorted(df_2016_2025["ms_name"].unique())

for t in thresholds:

    G = create_graph(aggregate_agreement, all_countries, threshold=t)

    partition = community_louvain.best_partition(
        G,
        weight="weight",
        random_state=42
    )

    modularity = community_louvain.modularity(
        partition,
        G,
        weight="weight"
    )

    sizes = Counter(partition.values())

    print("="*60)
    print(f"Threshold: {t}")
    print("Nodes:", G.number_of_nodes())
    print("Edges:", G.number_of_edges())
    print("Communities:", len(sizes))
    print("Modularity:", round(modularity,4))
    print("Community sizes:", sorted(sizes.values(), reverse=True))

In [ ]:
THRESHOLD = 0.70

In [ ]:
aggregate_G = create_graph(aggregate_agreement, all_countries, threshold=THRESHOLD)
gaza_G = create_graph(gaza_agreement, all_countries, threshold=THRESHOLD)
ukraine_G = create_graph(ukraine_agreement, all_countries, threshold=THRESHOLD)
climate_G = create_graph(climate_agreement, all_countries, threshold=THRESHOLD)

In [ ]:
from community import community_louvain

aggregate_partition = community_louvain.best_partition(
    aggregate_G,
    weight="weight",
    random_state=42
)

gaza_partition = community_louvain.best_partition(
    gaza_G,
    weight="weight",
    random_state=42
)

ukraine_partition = community_louvain.best_partition(
    ukraine_G,
    weight="weight",
    random_state=42
)

climate_partition = community_louvain.best_partition(
    climate_G,
    weight="weight",
    random_state=42
)

In [ ]:
from collections import Counter
from community import community_louvain

networks = {
    "Aggregate": (aggregate_G, aggregate_partition),
    "Gaza": (gaza_G, gaza_partition),
    "Ukraine": (ukraine_G, ukraine_partition),
    "Climate": (climate_G, climate_partition),
}

for name, (G, partition) in networks.items():

    modularity = community_louvain.modularity(
        partition,
        G,
        weight="weight"
    )

    sizes = Counter(partition.values())

    print("="*60)
    print(name)
    print("Nodes:", G.number_of_nodes())
    print("Edges:", G.number_of_edges())
    print("Communities:", len(sizes))
    print("Modularity:", round(modularity,4))
    print("Community sizes:", sorted(sizes.values(), reverse=True))

In [ ]:
weighted_degree = dict(aggregate_G.degree(weight="weight"))

In [ ]:
important = [

"UNITED STATES",
"RUSSIAN FEDERATION",
"CHINA",
"UNITED KINGDOM",
"FRANCE",

"INDIA",
"GERMANY",
"JAPAN",
"BRAZIL",

"PAKISTAN",

"CANADA",
"AUSTRALIA",
"ITALY",

"SOUTH AFRICA",
"INDONESIA",
"TURKEY",
"SAUDI ARABIA",

"IRAN (ISLAMIC REPUBLIC OF)",

"EGYPT",
"NIGERIA",

"MEXICO",
"ARGENTINA",

"REPUBLIC OF KOREA"

]

In [ ]:
pos = nx.spring_layout(
    aggregate_G,
    k=0.40,
    iterations=800,
    seed=42,
    weight="weight"
)

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import matplotlib.patheffects as pe

# -----------------------------
# Community colours
# -----------------------------
communities = sorted(set(aggregate_partition.values()))
color_map = plt.cm.Set2

community_colors = {
    c: color_map(i / max(1, len(communities)-1))
    for i, c in enumerate(communities)
}

# -----------------------------
# Node colours
# -----------------------------
node_colors = [
    community_colors[aggregate_partition[n]]
    for n in aggregate_G.nodes()
]

# -----------------------------
# Node sizes
# -----------------------------
node_sizes = []

for node in aggregate_G.nodes():

    if node in [
        "UNITED STATES",
        "RUSSIAN FEDERATION",
        "CHINA",
        "INDIA",
        "GERMANY",
        "JAPAN",
        "BRAZIL"
    ]:

        node_sizes.append(900)

    elif node in important:

        node_sizes.append(450)

    else:

        node_sizes.append(70)

# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(18,18))

nx.draw_networkx_edges(
    aggregate_G,
    pos,
    alpha=0.12,
    width=0.5,
    edge_color="grey"
)

nx.draw_networkx_nodes(
    aggregate_G,
    pos,
    node_color=node_colors,
    node_size=node_sizes,
    linewidths=0.3,
    edgecolors="black"
)

# -----------------------------
# Labels
# -----------------------------
for node in important:

    if node in pos:

        x,y = pos[node]

        if node in [
            "UNITED STATES",
            "RUSSIAN FEDERATION",
            "CHINA",
            "INDIA",
            "GERMANY",
            "JAPAN",
            "BRAZIL"
        ]:

            fontsize = 12

        else:

            fontsize = 9

        plt.text(
            x,
            y,
            node.title(),
            fontsize=fontsize,
            weight="bold",
            ha="center",
            va="center",
            path_effects=[
                pe.withStroke(
                    linewidth=3,
                    foreground="white"
                )
            ]
        )

plt.axis("off")

plt.title(
    "Aggregate UNGA Voting Network (Threshold = 0.70)",
    fontsize=18,
    weight="bold"
)

plt.savefig(
    "Figure_7_1_Aggregate.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
layouts = {}

for k in [0.4, 0.6, 0.8, 1.0]:

    layouts[k] = nx.spring_layout(
        aggregate_G,
        k=k,
        iterations=1500,
        seed=42,
        weight="weight"
    )

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(18,18))

for ax, k in zip(axes.flat, layouts):

    nx.draw_networkx_nodes(
        aggregate_G,
        layouts[k],
        node_size=20,
        node_color=[
            aggregate_partition[n]
            for n in aggregate_G.nodes()
        ],
        cmap="tab10",
        ax=ax
    )

    nx.draw_networkx_edges(
        aggregate_G,
        layouts[k],
        alpha=0.05,
        width=0.2,
        ax=ax
    )

    ax.set_title(f"k={k}")
    ax.axis("off")

plt.show()

In [ ]:
weights = [d["weight"] for _, _, d in aggregate_G.edges(data=True)]

print("Number of edges:", len(weights))
print("Minimum weight:", min(weights))
print("Maximum weight:", max(weights))

import numpy as np

for p in [50, 70, 80, 90, 95, 97, 99]:
    print(f"{p}th percentile:", np.percentile(weights, p))

In [ ]:
VISUAL_THRESHOLD = 0.95

aggregate_vis = nx.Graph()

aggregate_vis.add_nodes_from(aggregate_G.nodes())

for u, v, d in aggregate_G.edges(data=True):

    if d["weight"] >= VISUAL_THRESHOLD:

        aggregate_vis.add_edge(
            u,
            v,
            weight=d["weight"]
        )

print(
    aggregate_vis.number_of_nodes(),
    aggregate_vis.number_of_edges()
)

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import matplotlib.patheffects as pe

# ---------- Layout ----------
pos = nx.spring_layout(
    aggregate_vis,
    seed=42,
    k=0.85,
    iterations=2500,
    weight="weight"
)

# ---------- Community colours ----------
node_colors = [
    aggregate_partition[node]
    for node in aggregate_vis.nodes()
]

# ---------- Node sizes ----------
major = [
    "UNITED STATES",
    "RUSSIAN FEDERATION",
    "CHINA",
    "INDIA",
    "GERMANY",
    "JAPAN",
    "BRAZIL"
]

important = major + [
    "UNITED KINGDOM",
    "FRANCE",
    "PAKISTAN",
    "CANADA",
    "AUSTRALIA",
    "ITALY",
    "SOUTH AFRICA",
    "INDONESIA",
    "TURKEY",
    "SAUDI ARABIA",
    "IRAN (ISLAMIC REPUBLIC OF)",
    "EGYPT",
    "NIGERIA",
    "MEXICO",
    "ARGENTINA",
    "REPUBLIC OF KOREA"
]

sizes=[]

for n in aggregate_vis.nodes():

    if n in major:
        sizes.append(900)

    elif n in important:
        sizes.append(420)

    else:
        sizes.append(18)

# ---------- Figure ----------
plt.figure(figsize=(18,18))

nx.draw_networkx_edges(
    aggregate_vis,
    pos,
    alpha=0.08,
    width=0.3,
    edge_color="grey"
)

nx.draw_networkx_nodes(
    aggregate_vis,
    pos,
    node_size=sizes,
    node_color=node_colors,
    cmap=plt.cm.Set1,
    linewidths=0.4,
    edgecolors="black"
)

# ---------- Labels ----------
for n in important:

    if n not in pos:
        continue

    x,y=pos[n]

    fs=10

    if n in major:
        fs=13

    plt.text(
        x,
        y,
        n.title(),
        fontsize=fs,
        fontweight="bold",
        ha="center",
        va="center",
        path_effects=[
            pe.withStroke(
                linewidth=3,
                foreground="white"
            )
        ]
    )

plt.axis("off")

plt.tight_layout()

plt.savefig(
    "Figure7_1_Publication.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import networkx as nx

nx.write_gexf(aggregate_G, "aggregate_network.gexf")

from google.colab import files
files.download("aggregate_network.gexf")

In [ ]:
nx.write_gexf(gaza_G, "gaza_network.gexf")
nx.write_gexf(ukraine_G, "ukraine_network.gexf")
nx.write_gexf(climate_G, "climate_network.gexf")

from google.colab import files

files.download("gaza_network.gexf")
files.download("ukraine_network.gexf")
files.download("climate_network.gexf")

In [ ]:
for G in [aggregate_G, gaza_G, ukraine_G, climate_G]:
    for node in G.nodes():
        G.nodes[node]["label"] = node

import networkx as nx

nx.write_gexf(aggregate_G, "aggregate_network.gexf")
nx.write_gexf(gaza_G, "gaza_network.gexf")
nx.write_gexf(ukraine_G, "ukraine_network.gexf")
nx.write_gexf(climate_G, "climate_network.gexf")

In [ ]:
import os

print(os.listdir())

In [ ]:
from google.colab import files

files.download("aggregate_network.gexf")
files.download("gaza_network.gexf")
files.download("ukraine_network.gexf")
files.download("climate_network.gexf")